In [1]:
import sys
sys.path.append('..')

In [4]:
%load_ext autoreload
%autoreload 2
from src.join_signals import join_signals_to_spread, load_validated_pairs

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import pandas as pd
pairs_df = load_validated_pairs()
spread_series = pd.read_csv("../data/spread_series_2025_2026.csv", parse_dates=["date"])
signals_df = pd.read_csv("../data/extracted_signals.csv", parse_dates=["publish_date"])

### Initial Test Run (Small Sample Validation)

In [7]:
test_signals = signals_df.head(20)
test_result = join_signals_to_spread(test_signals, pairs_df, spread_series)
test_result

,ticker,paired_with,event_type_canonical,direction,confidence,event_date,zscore_before,zscore_after,zscore_change,expected_direction,direction_confirmed
0,2382.TW,4958.TW,strategic_partnership,1,0.75,2025-10-28 09:40:00,0.744629,0.726459,-0.018170,1,False
1,2356.TW,2492.TW,strategic_partnership,1,0.75,2025-10-28 09:40:00,-2.728500,-0.152068,2.576431,1,True


### Full Join: AI Signal vs. Spread Reaction (3-Day Window)

In [9]:
full_result = join_signals_to_spread(signals_df, pairs_df, spread_series)
full_result.to_csv("../data/event_spread_reactions.csv", index=False)
print(len(full_result))
full_result["direction_confirmed"].value_counts(dropna=False)

433


direction_confirmed
False    217
True     214
None       2
Name: count, dtype: int64

##### First Look: Direction Confirmation Rate ≈ 50% — No Detectable Signal
-----

### Robustness Check 1: Does Confidence Filtering Improve the Signal?

In [13]:
# 1. confidence filter
high_conf = full_result[full_result["confidence"] > 0.7]
high_conf["direction_confirmed"].value_counts(dropna=False)

direction_confirmed
False    211
True     210
Name: count, dtype: int64

### Robustness Check 2: Does Signal Strength Vary by Event Category?

In [14]:
# 2. by category
full_result.groupby("event_type_canonical")["direction_confirmed"].value_counts(normalize=True)

event_type_canonical            direction_confirmed
ai_demand                       True                   0.571429
                                False                  0.428571
analyst_and_market_sentiment    True                   0.512821
                                False                  0.487179
capacity_expansion              True                   0.600000
                                False                  0.400000
competitive_dynamics            False                  1.000000
corporate_events                False                  0.608696
                                True                   0.391304
financial_performance           True                   0.511111
                                False                  0.488889
geopolitical_and_regulatory     False                  1.000000
market_dynamics                 False                  0.576923
                                True                   0.423077
order_dynamics                  False               

### Robustness Check 3: Testing Alternative Reaction Windows (1-Day, 5-Day)

In [22]:
# 3. try different windows -- rerun join_signals_to_spread with window_days=1 and window_days=5
full_result_1day = join_signals_to_spread(signals_df, pairs_df, spread_series, window_days=1)
full_result_1day.to_csv("../data/event_spread_reactions_short.csv", index=False)
print(len(full_result_1day))
full_result_1day["direction_confirmed"].value_counts(dropna=False)

435


direction_confirmed
True     280
False    153
None       2
Name: count, dtype: int64

In [23]:
full_result_5day = join_signals_to_spread(signals_df, pairs_df, spread_series, window_days=5)
full_result_5day.to_csv("../data/event_spread_reactions_long.csv", index=False)
print(len(full_result_5day))
full_result_5day["direction_confirmed"].value_counts(dropna=False)

433


direction_confirmed
False    226
True     205
None       2
Name: count, dtype: int64

### Investigating the 1-Day Result: Naive Significance Test (and Why It's Misleading)

In [20]:
!pip install scipy
from scipy.stats import binomtest
result = binomtest(280, n=433, p=0.5)
print(result.pvalue)

  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
You should consider upgrading via the '/Users/for_everyoung10/Documents/ai-news-signal-extraction/venv/bin/python3 -m pip install --upgrade pip' command.
1.0736083150911588e-09


The binomial test against p=0.5 returns p≈1e-9, suggesting strong significance. 

However, this test assumes a "fair-coin" baseline where direction labels are unbiased (50/50) — but the actual label distribution is ~82% positive. 

A significant p-value here only tells us the result differs from this assumed baseline; it does not establish that the underlying relationship is real, since the baseline itself may be wrong. 

This motivated a proper test against the correct baseline.

-----

### Control Test 1: Random Dates + Random Directions

In [21]:
# 1-day-window analysis on random dates (not news-event dates)
import numpy as np

np.random.seed(42)  # reproducible

# tickers that actually appear in your validated pairs, so matches are possible
valid_tickers = pd.concat([pairs_df["stock1"], pairs_df["stock2"]]).unique()

n_fake = 300  # roughly similar scale to your real signal count

fake_signals_df = pd.DataFrame({
    "ticker": np.random.choice(valid_tickers, size=n_fake),
    "event_type_canonical": "random_control",  # placeholder, not used in the join logic
    "direction": np.random.choice([-1, 1], size=n_fake),  # skip 0, since neutral events get excluded from direction_confirmed anyway
    "confidence": 1.0,  # placeholder, not used unless you filter by it
    "publish_date": pd.to_datetime(np.random.choice(spread_series["date"].unique(), size=n_fake))
})

fake_result = join_signals_to_spread(fake_signals_df, pairs_df, spread_series, window_days=1)
fake_result["direction_confirmed"].value_counts(dropna=False)

direction_confirmed
True     254
False    243
Name: count, dtype: int64

### Control Test 2: Permutation Test (Preserving Real Event Dates and Direction Distribution)

In [24]:
n_permutations = 500
confirmed_rates = []

real_events_1day = signals_df[signals_df["ticker"].isin(valid_tickers)].copy()

for _ in range(n_permutations):
    shuffled = real_events_1day.copy()
    shuffled["direction"] = np.random.permutation(shuffled["direction"].values)
    perm_result = join_signals_to_spread(shuffled, pairs_df, spread_series, window_days=1)
    rate = perm_result["direction_confirmed"].mean()  # True=1, False=0, None excluded automatically by pandas mean
    confirmed_rates.append(rate)

real_rate = full_result_1day["direction_confirmed"].mean()  # from your actual 1-day result
print(f"Real rate: {real_rate:.3f}")
print(f"Permutation mean: {np.mean(confirmed_rates):.3f}, std: {np.std(confirmed_rates):.3f}")
print(f"Empirical p-value: {np.mean(np.array(confirmed_rates) >= real_rate):.4f}")

Real rate: 0.647
Permutation mean: 0.632, std: 0.015
Empirical p-value: 0.1840


#### Interpreting the Permutation Result: The True Null Baseline Is ~63%, Not 50%
The permutation test controls the parameter (events, dates, tickers, label distribution (82/18)) and only randomly assigned direction to events. 

The permutation mean is ~63% at 60-day window, and the real rate (64.7%) was statistically indistinguishable from the permutation distribution (p=0.184). 

This implies the AI-generated directions don't outperform randomly-assigned directions.

-----

### Final Robustness Check: Re-Running with a 20-Day Z-Score Window

In [25]:
spread_series_w20 = pd.read_csv("../data/spread_series_2025_2026_w20.csv", parse_dates=["date"])

result_w20_1day = join_signals_to_spread(signals_df, pairs_df, spread_series_w20, window_days=1)
real_rate_w20 = result_w20_1day["direction_confirmed"].mean()
print(f"Real rate (20-day zscore window): {real_rate_w20:.3f}")

# permutation test, same structure as before, on the w20 series
confirmed_rates_w20 = []
for _ in range(500):
    shuffled = real_events_1day.copy()
    shuffled["direction"] = np.random.permutation(shuffled["direction"].values)
    perm_result = join_signals_to_spread(shuffled, pairs_df, spread_series_w20, window_days=1)
    confirmed_rates_w20.append(perm_result["direction_confirmed"].mean())

print(f"Permutation mean: {np.mean(confirmed_rates_w20):.3f}, std: {np.std(confirmed_rates_w20):.3f}")
print(f"Empirical p-value: {np.mean(np.array(confirmed_rates_w20) >= real_rate_w20):.4f}")

Real rate (20-day zscore window): 0.584
Permutation mean: 0.584, std: 0.015
Empirical p-value: 0.5120


##### Robustness Confirmed: 20-Day Window Reproduces the Null Result Exactly

### Conclusion: No Statistically Significant Evidence That AI-Extracted Direction Predicts Spread Movement

The initial naive test of 1-day reaction window suggested strong significane, however, after digging deeper by conducting two independent control test (random-date, permutation), the original finding is refuted that the prediction pattern was driven by label-distribution skew instead of genuine predictive signal. 

Under rigorous testing, we can conclude that 60-day rolling z-score compute window with 1-day reaction window cannot support the original hypothesis that market news would provide meaningful impact on spreads movement between a pair of stocks. 

Even after changing the z-score window to 20 days, the same pattern held: the real confirmation rate (58.4%) matched the permutation baseline almost exactly (58.4% mean, p=0.512) — providing an independent confirmation that the earlier apparent signal was an artifact of label skew, not a specification-dependent fluke.

This doesn't mean AI is useless in statiscal pair trading arbitrage, it's just stating that a particular signal (ticker-level directional sentiment from news) doesn't add incremental predictive value to the existing price-based stat-arb model.